# 04 — Model Evaluation

Comprehensive evaluation of the APTOS fine-tuned model on the held-out test set:
- Quadratic Weighted Kappa (primary clinical metric)
- Per-class ROC curves with AUC
- Confusion matrix (normalized)
- Threshold optimization for improved grading
- Test-Time Augmentation (TTA) comparison
- Single-image inference demo


In [ ]:
import importlib
import os
import sys

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle scipy
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config


In [ ]:
import numpy as np
import pandas as pd
import torch

from src.augmentations import get_valid_transforms
from src.dataset import DRDataset, create_dataloaders, load_and_split_dataframe
from src.evaluate import (
    compute_metrics,
    full_evaluation,
    plot_training_curves,
    predict,
    predict_with_tta,
)
from src.model import build_model, count_parameters
from src.utils import get_device, seed_everything

seed_everything()
device = get_device()


## 1. Load Test Data and Model


In [ ]:
# Load APTOS test split (saved by 03b notebook or train_pipeline.py)
test_csv_path = os.path.join(config.OUTPUTS_DIR, "test_split_aptos.csv")
if os.path.isfile(test_csv_path):
    test_df = pd.read_csv(test_csv_path)
    print(f"Loaded APTOS test split: {len(test_df)} images")
else:
    # Fall back to creating a new split
    print("No saved test split found. Creating new APTOS split...")
    _, _, test_df = load_and_split_dataframe(dataset_name="aptos")
    print(f"Created APTOS test split: {len(test_df)} images")

print(f"\nTest class distribution:")
print(test_df["level"].value_counts().sort_index())


In [ ]:
# Build DataLoader
valid_transforms = get_valid_transforms(config.IMG_SIZE)

loaders = create_dataloaders(
    train_df=test_df,  # dummy — not used
    valid_df=test_df,
    dataset_name="aptos",
    valid_transform=valid_transforms,
    batch_size=config.STAGE2_BATCH_SIZE,
)
test_loader = loaders["valid"]

# Also create test dataset for single-image access
test_dataset = DRDataset(
    test_df,
    dataset_name="aptos",
    transform=valid_transforms,
    preprocessing=False,
)
print(f"Test loader: {len(test_loader)} batches")


In [ ]:
# Load trained model (APTOS fine-tuned)
model = build_model(pretrained=False)

ckpt_path = os.path.join(config.CHECKPOINTS_DIR, "best_model_aptos_finetune.pth")
if not os.path.isfile(ckpt_path):
    # Fall back to Stage 1 checkpoint
    ckpt_path = os.path.join(config.CHECKPOINTS_DIR, "best_model_s1_phase3.pth")
    print(f"APTOS checkpoint not found, using Stage 1: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device)
model.eval()

print(f"Loaded checkpoint: {os.path.basename(ckpt_path)}")
print(f"  Best QWK: {ckpt.get('qwk', 'N/A')}")
print(f"  Parameters: {count_parameters(model)['total']:,}")


## 2. Full Evaluation


In [ ]:
metrics = full_evaluation(
    model=model,
    test_loader=test_loader,
    device=device,
    test_dataset=test_dataset,
    save_dir=config.FIGURES_DIR,
)


## 3. Training Curves


In [ ]:
import json as json_mod

# Try Stage 2 history first, then Stage 1
for hist_name in ["stage2_history.json", "stage1_history.json", "training_history.json"]:
    history_path = os.path.join(config.LOGS_DIR, hist_name)
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json_mod.load(f)
        print(f"Loaded training history: {hist_name}")
        plot_training_curves(
            history,
            save_path=os.path.join(config.FIGURES_DIR, "training_curves.png"),
        )
        break
else:
    print("No training history found. Run training first.")


## 4. Single-Image Inference


In [ ]:
from src.evaluate import predict_single_image


def find_image(images_dir, image_id):
    """Try common extensions and return the file path, or None."""
    for ext in (".jpeg", ".jpg", ".png", ".JPEG", ".JPG"):
        p = os.path.join(images_dir, image_id + ext)
        if os.path.isfile(p):
            return p
    return None


# Pick one image per class from the test set
CLASS_CODE = dict(enumerate(config.CLASS_NAMES))

print("Single-image predictions (one per class):\n")
for cls in sorted(test_df["level"].unique()):
    sample = test_df[test_df["level"] == cls].iloc[0]
    img_path = find_image(config.APTOS_IMAGES_DIR, sample["image"])
    if img_path is None:
        # Try preprocessed cache
        cache_dir = os.path.join(config.APTOS_DATA_DIR, f"preprocessed_{config.IMG_SIZE}_aptos")
        img_path = find_image(cache_dir, sample["image"])
    if img_path is None:
        print(f"  Class {cls} ({CLASS_CODE[cls]}): image not found")
        continue

    pred_class, proba = predict_single_image(model, img_path, device)
    true_name = CLASS_CODE[cls]
    pred_name = CLASS_CODE[pred_class]
    correct = "✅" if pred_class == cls else "❌"
    print(f"  True: {true_name:20s} → Pred: {pred_name:20s} {correct}  (conf: {proba[pred_class]:.3f})")


## 5. Threshold Optimization

Standard argmax assigns to the highest-probability class. But ordinal class
boundaries can be improved by optimizing thresholds to maximize QWK.


In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import cohen_kappa_score

# Get predictions
y_true, y_pred, y_proba = predict(model, test_loader, device)

# Convert probabilities to continuous expected value
y_proba_np = y_proba.numpy() if isinstance(y_proba, torch.Tensor) else np.array(y_proba)
y_continuous = (y_proba_np * np.arange(5)).sum(axis=1)

# Optimize thresholds
def neg_qwk(thresholds):
    t = sorted(thresholds)
    preds = np.digitize(y_continuous, t)
    return -cohen_kappa_score(y_true, preds, weights="quadratic")

result = minimize(neg_qwk, [0.5, 1.5, 2.5, 3.5], method="Nelder-Mead")
opt_thresholds = sorted(result.x)
y_pred_opt = np.digitize(y_continuous, opt_thresholds)

qwk_argmax = cohen_kappa_score(y_true, y_pred, weights="quadratic")
qwk_opt = cohen_kappa_score(y_true, y_pred_opt, weights="quadratic")

print(f"Optimal thresholds: {[f'{t:.3f}' for t in opt_thresholds]}")
print(f"\nQWK (argmax):              {qwk_argmax:.4f}")
print(f"QWK (optimized thresholds): {qwk_opt:.4f}")
print(f"Improvement:                {qwk_opt - qwk_argmax:+.4f}")


## 6. Test-Time Augmentation (TTA)

TTA averages predictions across multiple geometric transforms of the same image,
typically improving QWK by 0.5–2%.


In [ ]:
# Standard prediction (no TTA)
y_true, y_pred_std, y_proba_std = predict(model, test_loader, device)
metrics_std = compute_metrics(y_true, y_pred_std, y_proba_std)

# TTA prediction (4 augmented views)
y_true_tta, y_pred_tta, y_proba_tta = predict_with_tta(model, test_loader, device, n_augments=4)
metrics_tta = compute_metrics(y_true_tta, y_pred_tta, y_proba_tta)

print("Standard vs TTA comparison:")
print(f"  {'Metric':<20s} {'Standard':>10s} {'TTA (n=4)':>10s} {'Δ':>8s}")
print(f"  {'-'*50}")
for metric in ["qwk", "accuracy", "roc_auc_weighted"]:
    s = metrics_std[metric]
    t = metrics_tta[metric]
    print(f"  {metric:<20s} {s:>10.4f} {t:>10.4f} {t-s:>+8.4f}")


## 7. Summary

Key metrics to report:
- **QWK** (Quadratic Weighted Kappa): the gold standard for DR grading
- **Per-class ROC-AUC**: how well each severity level is discriminated
- **Threshold-optimized QWK**: potential improvement with careful class boundaries
- **TTA QWK**: improvement from multi-view prediction averaging
- **Grad-CAM** (see notebook 05): visual explanations for clinical trust
